In [3]:
#!/usr/bin/env python3
import os
import json
import random
import torch
import open_clip
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# -------------------- Configuration --------------------
ROOT_DIR         = ""  # Data root directory
DATA_JSON        = os.path.join(ROOT_DIR, "dataset_rsicd.json")
RSICD_IMG_DIR    = os.path.join(ROOT_DIR, "RSICD_images")
LEADERBOARD_DIR  = os.path.join(ROOT_DIR, "LEADERBOARD_DIR")
BEST_MODEL_PATH  = "improved_clip.pth"
PSEUDO_JSON      = "pseudo_labels.json"

# Hyperparameters
PSEUDO_THRESHOLD = 0.5   # Threshold for pseudo-label confidence
TOP_K            = 5     # Number of top candidates for pseudo labels
BATCH_SIZE       = 16    # Batch size
EPOCHS_FINETUNE  = 10    # Number of finetuning epochs
SUP_WEIGHT       = 1.0   # Supervised loss weight
PSEUDO_WEIGHT    = 0.1   # Pseudo-label loss weight
TEMPERATURE      = 0.2   # Temperature for retrieval

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Multi-template list
TEMPLATES = [
    "a satellite photo of {c}",
    "an aerial view showing {c}",
    "high-resolution satellite image of {c}",
    "detailed remote sensing image of {c}",
    "orthophoto capturing {c}",
    "satellite imagery of {c}"
]


def sample_prompts(captions):
    """Randomly sample a template for each caption"""
    return [random.choice(TEMPLATES).format(c=c) for c in captions]


def recompute_caption_embeddings(model, tokenizer, ann):
    """Compute mean text embedding prototypes for validation images"""
    image_captions, image_ids = {}, []
    for img in ann["images"]:
        if img.get("split") == "val":
            iid = img.get("imgid", img.get("filename"))
            image_ids.append(iid)
            image_captions[iid] = [s["raw"] for s in img["sentences"]]
    if not image_ids:
        raise ValueError("No validation images found. Check 'split' values in dataset_rsicd.json.")
    mean_embs = []
    with torch.no_grad():
        for iid in tqdm(image_ids, desc="Computing prototypes"):
            caps = image_captions[iid]
            prompts = sample_prompts(caps)
            toks = tokenizer(prompts).to(DEVICE)
            txt = model.encode_text(toks)
            txt = torch.nn.functional.normalize(txt, dim=-1)
            m = txt.mean(dim=0, keepdim=True)
            m = torch.nn.functional.normalize(m, dim=-1)
            mean_embs.append(m)
    return torch.cat(mean_embs, 0), image_ids, image_captions


def generate_pseudo_labels(model, preprocess, mean_embs, image_ids, image_captions):
    """Generate pseudo-labels for unlabeled images"""
    ann = json.load(open(DATA_JSON))
    rsicd_unlab = [img["filename"] for img in ann["images"] if img["split"] == "train"]
    lb_files = [f for f in os.listdir(LEADERBOARD_DIR) if f.lower().endswith(".jpg")] if os.path.exists(LEADERBOARD_DIR) else []
    all_files = rsicd_unlab + lb_files
    if not all_files:
        raise ValueError("No unlabeled images found.")
    pseudo, valid_count = {}, 0
    with torch.no_grad():
        for i in tqdm(range(0, len(all_files), BATCH_SIZE), desc="Gen pseudo-labels"):
            batch = all_files[i:i+BATCH_SIZE]
            imgs, names = [], []
            for fn in batch:
                path = os.path.join(RSICD_IMG_DIR, fn)
                if not os.path.exists(path): path = os.path.join(LEADERBOARD_DIR, fn)
                if not os.path.exists(path): continue
                img = Image.open(path).convert("RGB")
                imgs.append(preprocess(img).unsqueeze(0).to(DEVICE))
                names.append(fn)
            if not imgs: continue
            imgs = torch.cat(imgs, 0)
            img_emb = model.encode_image(imgs)
            img_emb = torch.nn.functional.normalize(img_emb, dim=-1)
            logits = 100 * (img_emb @ mean_embs.T)
            probs = logits.softmax(dim=-1)
            topv, topi = probs.topk(TOP_K, dim=-1)
            for bi, fn in enumerate(names):
                pseudo[fn] = []
                for k in range(TOP_K):
                    conf = topv[bi, k].item()
                    if conf < PSEUDO_THRESHOLD: continue
                    sid = image_ids[topi[bi, k].item()]
                    caps = image_captions[sid]
                    prompts = sample_prompts(caps)
                    toks = tokenizer(prompts).to(DEVICE)
                    txt_emb = model.encode_text(toks)
                    txt_emb = torch.nn.functional.normalize(txt_emb, dim=-1)
                    scores = (txt_emb @ img_emb[bi:bi+1].T).squeeze()
                    best = caps[scores.argmax().item()]
                    pseudo[fn].append({"caption": best, "confidence": conf, "source_img_id": sid})
                if pseudo[fn]: valid_count += 1
    print(f"Generated pseudo-labels for {valid_count}/{len(all_files)} images.")
    with open(PSEUDO_JSON, 'w', encoding='utf-8') as f:
        json.dump(pseudo, f, indent=2, ensure_ascii=False)
    return pseudo


class PseudoDataset(Dataset):
    def __init__(self, pseudo_labels, preprocess):
        self.data = [(fn, caps) for fn, caps in pseudo_labels.items() if caps]
        self.prep = preprocess
        if not self.data: print("Warning: PseudoDataset empty.")
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        fn, caps = self.data[idx]
        path = os.path.join(RSICD_IMG_DIR, fn)
        if not os.path.exists(path): path = os.path.join(LEADERBOARD_DIR, fn)
        img = Image.open(path).convert("RGB")
        return self.prep(img), caps


class LabeledDataset(Dataset):
    def __init__(self, ann, preprocess):
        self.data = [(img["filename"], [s["raw"] for s in img["sentences"]])
                     for img in ann["images"] if img["split"] == "train"]
        self.prep = preprocess
        if not self.data: print("Warning: LabeledDataset empty.")
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        fn, caps = self.data[idx]
        img = Image.open(os.path.join(RSICD_IMG_DIR, fn)).convert("RGB")
        return self.prep(img), caps


def compute_val_metrics(model, preprocess, tokenizer, ann):
    """Compute retrieval metrics on val set"""
    data = [e for e in ann["images"] if e["split"] == "val"]
    if not data: return 0.0,0.0,0.0
    feats, names = [], []
    with torch.no_grad():
        for i in range(0,len(data),BATCH_SIZE):
            batch = data[i:i+BATCH_SIZE]
            imgs=[]
            for e in batch:
                img=Image.open(os.path.join(RSICD_IMG_DIR,e["filename"])).convert("RGB")
                imgs.append(preprocess(img).unsqueeze(0)); names.append(e["filename"])
            imgs=torch.cat(imgs,0).to(DEVICE)
            f=model.encode_image(imgs); feats.append(f/f.norm(dim=-1,keepdim=True))
    img_mat=torch.cat(feats,0)
    R1=R5=R10=0
    for e in data:
        cap_embs=[]
        for c in [s["raw"] for s in e["sentences"]]:
            prompts=[tmpl.format(c=c) for tmpl in TEMPLATES]
            toks=tokenizer(prompts).to(DEVICE)
            txt=model.encode_text(toks)
            txt=txt/txt.norm(dim=-1,keepdim=True)
            mean_txt=txt.mean(dim=0,keepdim=True)
            cap_embs.append(mean_txt)
        sims=(torch.cat(cap_embs,0) @ img_mat.T)/TEMPERATURE
        sims=sims.mean(0)
        topk=sims.topk(10).indices.cpu().tolist()
        preds=[names[i] for i in topk]; gt=e["filename"]
        R1+=int(preds[0]==gt); R5+=int(gt in preds[:5]); R10+=int(gt in preds[:10])
    tot=len(data)
    return R1/tot*100, R5/tot*100, R10/tot*100


def collate_fn(batch):
    imgs, caps = [], []
    for img, cap_list in batch:
        img = img.unsqueeze(0)
        for cap in cap_list:
            imgs.append(img)
            if isinstance(cap, dict):
                caps.append(cap["caption"])
            else:
                caps.append(cap)
    return torch.cat(imgs, 0), caps

def train_joint(model, preprocess, tokenizer, pseudo, ann):
    """Fine-tune with supervised and pseudo-label loss"""
    sup_ds = LabeledDataset(ann, preprocess)
    pseudo_ds = PseudoDataset(pseudo, preprocess)
    sup_loader = DataLoader(sup_ds, batch_size=BATCH_SIZE//2,shuffle=True,collate_fn=lambda b:collate_fn(b))
    pseudo_loader = DataLoader(pseudo_ds,batch_size=BATCH_SIZE//2,shuffle=True,collate_fn=lambda b:collate_fn(b))
    optim = AdamW([{'params':[p for n,p in model.named_parameters() if p.requires_grad and 'visual' in n],'lr':5e-6},
                   {'params':[p for n,p in model.named_parameters() if p.requires_grad and ('text_projection' in n or 'logit_scale' in n)],'lr':5e-5}],
                  betas=(0.9,0.98),weight_decay=0.1)
    total_steps=EPOCHS_FINETUNE*min(len(sup_loader),len(pseudo_loader))
    sched=OneCycleLR(optim,max_lr=[5e-6,5e-5],total_steps=total_steps,pct_start=0.1,anneal_strategy='cos',final_div_factor=1e6)
    loss_fn=open_clip.loss.ClipLoss()
    best=0
    for epoch in range(1,EPOCHS_FINETUNE+1):
        model.train()
        it_s, it_p = iter(sup_loader), iter(pseudo_loader)
        for _ in range(min(len(sup_loader),len(pseudo_loader))):
            imgs_s,caps_s=next(it_s); imgs_p,caps_p=next(it_p)
            # supervised
            toks_s=tokenizer(sample_prompts(caps_s)).to(DEVICE)
            img_emb_s=model.encode_image(imgs_s.to(DEVICE)); txt_emb_s=model.encode_text(toks_s)
            img_emb_s=img_emb_s/img_emb_s.norm(dim=-1,keepdim=True)
            txt_emb_s=txt_emb_s/txt_emb_s.norm(dim=-1,keepdim=True)
            loss_s=loss_fn(img_emb_s,txt_emb_s,model.logit_scale)
            # pseudo
            toks_p=tokenizer(sample_prompts(caps_p)).to(DEVICE)
            img_emb_p=model.encode_image(imgs_p.to(DEVICE)); txt_emb_p=model.encode_text(toks_p)
            img_emb_p=img_emb_p/img_emb_p.norm(dim=-1,keepdim=True)
            txt_emb_p=txt_emb_p/txt_emb_p.norm(dim=-1,keepdim=True)
            loss_p=loss_fn(img_emb_p,txt_emb_p,model.logit_scale)
            loss=SUP_WEIGHT*loss_s+PSEUDO_WEIGHT*loss_p
            optim.zero_grad(); loss.backward(); optim.step(); sched.step()
        model.eval()
        r1,_,_=compute_val_metrics(model,preprocess,tokenizer,ann)
        if r1>best:
            best=r1; torch.save(model.state_dict(),BEST_MODEL_PATH)
            print(f"[Epoch {epoch}] New best R@1={r1:.2f}% saved.")


def text_to_image_retrieval(model, preprocess, tokenizer, split="test"):
    """Evaluate text→image using multi-template fusion"""
    data=json.load(open(DATA_JSON))["images"]
    test=[e for e in data if e.get("split")==split]
    feats,names=[],[]
    with torch.no_grad():
        for i in range(0,len(test),BATCH_SIZE):
            batch=test[i:i+BATCH_SIZE]; imgs=[]
            for e in batch:
                img=Image.open(os.path.join(RSICD_IMG_DIR,e["filename"])).convert("RGB")
                imgs.append(preprocess(img).unsqueeze(0)); names.append(e["filename"])
            imgs=torch.cat(imgs,0).to(DEVICE)
            f=model.encode_image(imgs); feats.append(f/f.norm(dim=-1,keepdim=True))
    img_mat=torch.cat(feats,0)
    R1=R5=R10=0
    for e in tqdm(test,desc="Retrieving"):
        cap_embs=[]
        for c in [s["raw"] for s in e["sentences"]]:
            prompts=[tmpl.format(c=c) for tmpl in TEMPLATES]
            toks=tokenizer(prompts).to(DEVICE)
            txt=model.encode_text(toks)
            txt=txt/txt.norm(dim=-1,keepdim=True)
            m=txt.mean(dim=0,keepdim=True)
            cap_embs.append(m)
        sims=(torch.cat(cap_embs,0) @ img_mat.T)/TEMPERATURE
        sims=sims.mean(0)
        idxs=sims.topk(10).indices.cpu().tolist(); preds=[names[i] for i in idxs]
        gt=e["filename"]; R1+=int(preds[0]==gt); R5+=int(gt in preds[:5]); R10+=int(gt in preds[:10])
    tot=len(test)
    print(f"\nText→Image on {tot} queries: R@1={R1/tot*100:.2f}%, R@5={R5/tot*100:.2f}%, R@10={R10/tot*100:.2f}%")
    return R1/tot*100, R5/tot*100, R10/tot*100

if __name__ == "__main__":
    # Load model and tokenizer
    model, _, base_preprocess = open_clip.create_model_and_transforms(
        #"ViT-B-16", pretrained="laion400m_e31"
        "ViT-B-32", pretrained="laion400m_e31"
    )
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    preprocess = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
        transforms.RandomHorizontalFlip(0.3),
        transforms.ColorJitter(0.1, 0.1, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.48145466,0.4578275,0.40821073), std=(0.26862954,0.26130258,0.27577711))
    ])
    model = model.to(DEVICE).eval()

    # Zero-shot evaluation
    print("Zero-shot test:")
    text_to_image_retrieval(model, base_preprocess, tokenizer, split="test")

    # Pseudo-labeling
    ann = json.load(open(DATA_JSON))
    mean_embs, img_ids, img_caps = recompute_caption_embeddings(model, tokenizer, ann)
    pseudo = generate_pseudo_labels(model, base_preprocess, mean_embs, img_ids, img_caps)

    # Fine-tuning
    train_joint(model, preprocess, tokenizer, pseudo, ann)

    # Final evaluation
    print("Post-finetune test:")
    text_to_image_retrieval(model, base_preprocess, tokenizer, split="test")


open_clip_model.safetensors:  94%|#########3| 566M/605M [00:00<?, ?B/s]

/home/leime957/.local/lib/python3.12/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'laion400m_e31' (quick_gelu=True).
  warnings.warn(


Zero-shot test:


Retrieving: 100%|███████████████████████████████████████████████| 1093/1093 [00:29<00:00, 36.50it/s]



Text→Image on 1093 queries: R@1=7.41%, R@5=21.59%, R@10=33.85%


Gen pseudo-labels: 100%|██████████████████████████████████████████| 546/546 [00:24<00:00, 22.64it/s]


Generated pseudo-labels for 357/8734 images.
[Epoch 1] New best R@1=6.95% saved.
[Epoch 2] New best R@1=8.04% saved.
[Epoch 4] New best R@1=8.59% saved.
[Epoch 5] New best R@1=9.96% saved.
Post-finetune test:


Retrieving: 100%|███████████████████████████████████████████████| 1093/1093 [00:30<00:00, 35.61it/s]


Text→Image on 1093 queries: R@1=13.17%, R@5=36.14%, R@10=54.44%
